# Classifier Chains & Ensemble of Classifier Chains

**Dataset:** GoodScents + Leffingwell  
**Preprocessing:** identical to baseline.ipynb (same split, same scaling, same correlation filter)

We compare two chain-based strategies against Binary Relevance:
- **CC** — single Classifier Chain (fixed random label order)
- **ECC** — Ensemble of Classifier Chains (10 chains, different random orderings, majority vote)

## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from skmultilearn.model_selection import iterative_train_test_split

## Load & Preprocess Data

Identical pipeline to baseline.ipynb.

In [2]:
df = pd.read_csv('goodscents_jadbio_ready.csv', sep=';')

LABEL_COLS = [
    'floral', 'fruity', 'sweet', 'woody', 'green', 'spicy',
    'animal_musk', 'earthy', 'citrus', 'chemical', 'gourmand', 'powdery_amber'
]

fp_cols      = [c for c in df.columns if c.startswith('MACCS_') or c.startswith('morgan_')]
mordred_cols = [c for c in df.columns if c not in LABEL_COLS + ['SMILES'] + fp_cols]

df_clean = df.dropna().reset_index(drop=True)

X_fp      = df_clean[fp_cols].values.astype(float)
X_mordred = df_clean[mordred_cols].values.astype(float)
y         = df_clean[LABEL_COLS].values

# Zero-variance filter
vt_fp = VarianceThreshold(threshold=0)
X_fp  = vt_fp.fit_transform(X_fp)

vt_mordred = VarianceThreshold(threshold=0)
X_mordred  = vt_mordred.fit_transform(X_mordred)

# Train/test split (iterative stratified)
X_combined = np.hstack([X_fp, X_mordred])
n_fp = X_fp.shape[1]

X_train_comb, y_train, X_test_comb, y_test = iterative_train_test_split(
    X_combined, y, test_size=0.2
)

X_fp_train    = X_train_comb[:, :n_fp]
X_mordred_train = X_train_comb[:, n_fp:]
X_fp_test     = X_test_comb[:, :n_fp]
X_mordred_test  = X_test_comb[:, n_fp:]

# Scale Mordred (fit on train only)
scaler = StandardScaler()
X_mordred_train = scaler.fit_transform(X_mordred_train)
X_mordred_test  = scaler.transform(X_mordred_test)

# Correlation filter on Mordred (train only)
corr_matrix    = np.abs(np.corrcoef(X_mordred_train.T))
upper_triangle = np.triu(corr_matrix, k=1)
cols_to_drop   = set()
for r, c in zip(*np.where(upper_triangle > 0.95)):
    if c not in cols_to_drop:
        cols_to_drop.add(c)
cols_to_keep    = [i for i in range(X_mordred_train.shape[1]) if i not in cols_to_drop]
X_mordred_train = X_mordred_train[:, cols_to_keep]
X_mordred_test  = X_mordred_test[:, cols_to_keep]

X_train = np.hstack([X_fp_train, X_mordred_train])
X_test  = np.hstack([X_fp_test,  X_mordred_test])

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}')
print(f'y_test  : {y_test.shape}')

X_train : (3862, 993)
X_test  : (1114, 993)
y_train : (3862, 12)
y_test  : (1114, 12)


## Shared Evaluation Helper

In [3]:
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    balanced_accuracy_score, matthews_corrcoef,
    recall_score, confusion_matrix
)

def evaluate(name, y_te, y_pred, y_prob, label_cols):
    rows = []
    for i, label in enumerate(label_cols):
        yt, yp, ypr = y_te[:, i], y_pred[:, i], y_prob[:, i]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
        try:    roc = roc_auc_score(yt, ypr)
        except: roc = float('nan')
        try:    pr = average_precision_score(yt, ypr)
        except: pr = float('nan')
        rows.append({
            'Model'       : name,
            'Label'       : label,
            'Bal.Acc'     : round(balanced_accuracy_score(yt, yp), 3),
            'MCC'         : round(matthews_corrcoef(yt, yp), 3),
            'F1'          : round(f1_score(yt, yp, zero_division=0), 3),
            'ROC_AUC'     : round(roc, 3),
            'PR_AUC'      : round(pr, 3),
            'Sensitivity' : round(recall_score(yt, yp, zero_division=0), 3),
            'Specificity' : round(tn / (tn + fp) if (tn + fp) > 0 else float('nan'), 3),
        })
    return pd.DataFrame(rows)


def print_results(df_res, name):
    print(f'\n=== {name} ===')
    print(f'{"Label":<20} {"Bal.Acc":>8} {"MCC":>8} {"F1":>8} {"ROC_AUC":>8} {"PR_AUC":>8} {"Sensitivity":>12} {"Specificity":>12}')
    print('-' * 96)
    for _, r in df_res.iterrows():
        print(f'{r["Label"]:<20} {r["Bal.Acc"]:>8.3f} {r["MCC"]:>8.3f} {r["F1"]:>8.3f} {r["ROC_AUC"]:>8.3f} {r["PR_AUC"]:>8.3f} {r["Sensitivity"]:>12.3f} {r["Specificity"]:>12.3f}')
    print('-' * 96)
    macro = df_res.drop(columns=['Model', 'Label']).mean()
    print(f'{"MACRO AVG":<20} {macro["Bal.Acc"]:>8.3f} {macro["MCC"]:>8.3f} {macro["F1"]:>8.3f} {macro["ROC_AUC"]:>8.3f} {macro["PR_AUC"]:>8.3f} {macro["Sensitivity"]:>12.3f} {macro["Specificity"]:>12.3f}')

# Model: Classifier Chain + Random Forest

We use RF as the base classifier (best performer in the baseline comparison).

The label order is randomly shuffled — CC is sensitive to ordering, so we fix a seed for reproducibility.
The chain trains 12 classifiers sequentially:
- Classifier 1: predicts label_order[0] from X
- Classifier 2: predicts label_order[1] from X + predicted label_order[0]
- ...
- Classifier 12: predicts label_order[11] from X + all 11 previous predictions

In [4]:
from sklearn.multioutput import ClassifierChain
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

scorer = make_scorer(f1_score, average='macro')

# ClassifierChain from sklearn wraps any base estimator
# order='random' + random_state fixes the label permutation
cc_rf = ClassifierChain(
    base_estimator=RandomForestClassifier(
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    order='random',
    random_state=42
)

# Note: GridSearchCV param prefix is 'base_estimator__' for ClassifierChain
gs_cc = GridSearchCV(
    cc_rf,
    {'base_estimator__n_estimators': [100, 300],
     'base_estimator__max_features': ['sqrt', 'log2']},
    scoring=scorer, cv=3, n_jobs=-1, verbose=1
)

print('Fitting CC + RF...')
gs_cc.fit(X_train, y_train)
print(f'Best params : {gs_cc.best_params_}  |  CV macro-F1 : {gs_cc.best_score_:.3f}')

best_cc = gs_cc.best_estimator_
y_pred_cc = best_cc.predict(X_test)
y_prob_cc = best_cc.predict_proba(X_test)

# ClassifierChain returns predictions in chain order — reorder back to LABEL_COLS order
y_pred_cc = y_pred_cc[:, np.argsort(best_cc.order_)]
y_prob_cc = y_prob_cc[:, np.argsort(best_cc.order_)]

results_cc = evaluate('CC_RF', y_test, y_pred_cc, y_prob_cc, LABEL_COLS)
print_results(results_cc, 'Classifier Chain + RF')

Fitting CC + RF...


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


Fitting 3 folds for each of 4 candidates, totalling 12 fits


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


Best params : {'base_estimator__max_features': 'sqrt', 'base_estimator__n_estimators': 300}  |  CV macro-F1 : 0.475


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)



=== Classifier Chain + RF ===
Label                 Bal.Acc      MCC       F1  ROC_AUC   PR_AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.630    0.232    0.439    0.659    0.325        0.538        0.721
fruity                  0.460   -0.159    0.030    0.336    0.313        0.018        0.903
sweet                   0.520    0.059    0.222    0.568    0.393        0.151        0.890
woody                   0.417   -0.173    0.040    0.257    0.123        0.038        0.795
green                   0.529    0.142    0.137    0.614    0.529        0.076        0.982
spicy                   0.544    0.070    0.307    0.578    0.222        0.474        0.613
animal_musk             0.504    0.009    0.122    0.419    0.132        0.104        0.904
earthy                  0.543    0.066    0.298    0.554    0.195        0.538        0.549
citrus                  0.551    0.094    0.

# Model: Ensemble of Classifier Chains + Random Forest

ECC trains N chains, each with a **different random label ordering**.
Final prediction for each label = **majority vote** across the N chains.
Probability = **average predicted probability** across chains.

This reduces the sensitivity to any single ordering and generally outperforms a single CC.
We use N=10 chains — a standard choice from Read et al. (2011).

In [5]:
N_CHAINS = 10

# Use the best RF params found above
best_n_est     = gs_cc.best_params_['base_estimator__n_estimators']
best_max_feat  = gs_cc.best_params_['base_estimator__max_features']

chains = [
    ClassifierChain(
        base_estimator=RandomForestClassifier(
            n_estimators=best_n_est,
            max_features=best_max_feat,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ),
        order='random',
        random_state=seed
    )
    for seed in range(N_CHAINS)
]

print(f'Training {N_CHAINS} chains...')
for i, chain in enumerate(chains):
    chain.fit(X_train, y_train)
    print(f'  Chain {i+1}/{N_CHAINS} done')

# Collect predictions from each chain, reordered back to LABEL_COLS
all_preds = np.array([
    chain.predict(X_test)[:, np.argsort(chain.order_)]
    for chain in chains
])  # shape: (N_CHAINS, n_samples, n_labels)

all_probs = np.array([
    chain.predict_proba(X_test)[:, np.argsort(chain.order_)]
    for chain in chains
])  # shape: (N_CHAINS, n_samples, n_labels)

# Majority vote for labels, mean for probabilities
y_pred_ecc = (all_preds.mean(axis=0) >= 0.5).astype(int)
y_prob_ecc = all_probs.mean(axis=0)

results_ecc = evaluate('ECC_RF', y_test, y_pred_ecc, y_prob_ecc, LABEL_COLS)
print_results(results_ecc, 'Ensemble of Classifier Chains + RF')

Training 10 chains...


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


  Chain 1/10 done


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


  Chain 2/10 done


...

  Chain 9/10 done


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


  Chain 10/10 done


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\h


=== Ensemble of Classifier Chains + RF ===
Label                 Bal.Acc      MCC       F1  ROC_AUC   PR_AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.488   -0.057    0.027    0.475    0.204        0.015        0.960
fruity                  0.528    0.101    0.188    0.558    0.438        0.112        0.944
sweet                   0.543    0.134    0.250    0.677    0.518        0.164        0.923
woody                   0.510    0.031    0.125    0.494    0.183        0.085        0.935
green                   0.549    0.115    0.362    0.661    0.527        0.280        0.818
spicy                   0.478   -0.075    0.036    0.470    0.167        0.023        0.932
animal_musk             0.531    0.065    0.186    0.581    0.193        0.175        0.886
earthy                  0.494   -0.025    0.042    0.610    0.204        0.025        0.963
citrus                  0.499  

# Summary: CC vs ECC vs BR (RF baseline)

BR+RF results are pasted from baseline.ipynb for direct comparison.

In [6]:
# BR+RF macro results from baseline.ipynb
br_rf_macro = {
    'Model': 'BR_RF',
    'Bal.Acc': 0.689, 'MCC': 0.432, 'F1': 0.546,
    'ROC_AUC': 0.820, 'PR_AUC': 0.624,
    'Sensitivity': 0.487, 'Specificity': 0.892
}

metric_cols = ['Bal.Acc', 'MCC', 'F1', 'ROC_AUC', 'PR_AUC', 'Sensitivity', 'Specificity']

all_results = pd.concat([results_cc, results_ecc], ignore_index=True)
summary = (
    all_results.groupby('Model')[metric_cols]
    .mean()
    .round(3)
    .reset_index()
)

# Add BR+RF row
summary = pd.concat([summary, pd.DataFrame([br_rf_macro])], ignore_index=True)
summary = summary.set_index('Model').loc[['BR_RF', 'CC_RF', 'ECC_RF']]

print('Macro-averaged test metrics: BR vs CC vs ECC (RF base)')
print(summary.to_string())

Macro-averaged test metrics: BR vs CC vs ECC (RF base)
        Bal.Acc    MCC     F1  ROC_AUC  PR_AUC  Sensitivity  Specificity
Model                                                                   
BR_RF     0.689  0.432  0.546    0.820   0.624        0.487        0.892
CC_RF     0.508  0.011  0.196    0.487   0.256        0.225        0.791
ECC_RF    0.515  0.037  0.146    0.544   0.286        0.103        0.926
